In [4]:
import pyspark
from pyspark.sql import SparkSession
import pandas as pd

In [7]:
spark = (
    SparkSession.builder \
    .master("local[1]") \
    .appName("zoomcamp") \
    .config("spark.driver.memory", "512m") \
    .config("spark.executor.memory", "512m") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()
)

In [6]:
spark.stop()

In [11]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-05 23:50:49--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.198, 3.170.186.229, 3.170.186.41, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.198|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet.1’

yellow_tripdata_202 100%[===================>]  67.84M  99.5MB/s    in 0.7s    

2026-03-05 23:50:50 (99.5 MB/s) - ‘yellow_tripdata_2025-11.parquet.1’ saved [71134255/71134255]



In [8]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [9]:
df.printSchema()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
df = df.repartition(4)

In [12]:
df.write.parquet('data_yellow/')

In [ ]:
# 4 files of size 27mb

In [13]:
df.count()

4181444

In [ ]:
# 4181444 in the spark dataframe

In [17]:
from pyspark.sql import functions as F

In [19]:
df.filter(F.day(df.tpep_pickup_datetime) == 15).count()

162604

In [ ]:
# November 15 rows:  162604

In [32]:
df.select(F.timestamp_diff("hour", df.tpep_pickup_datetime, df.tpep_dropoff_datetime).alias("traveltime")).select(F.max("traveltime")).show()


[Stage 31:===========================================>              (3 + 1) / 4]

+---------------+
|max(traveltime)|
+---------------+
|             90|
+---------------+



In [ ]:
# Max travel time = 90 hours (floored)

In [33]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-06 03:54:36--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.170.186.111, 3.170.186.41, 3.170.186.198, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.170.186.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-06 03:54:36 (143 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [35]:
lookup_df = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')

In [36]:
lookup_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [40]:
df_join = df.join(lookup_df, df.PULocationID == lookup_df.LocationID, how='inner')

In [41]:
df_join.registerTempTable('zoned')

/home/bb-devbot/.venv/lib/python3.11/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [45]:
report = spark.sql("""
SELECT 
    Borough, 
    Zone,
    COUNT(1) 
FROM
    zoned
GROUP BY
    1, 2
ORDER BY 3 asc
LIMIT 20
""")

In [46]:
report.show()

[Stage 47:===========================================>              (3 + 1) / 4]

+-------------+--------------------+--------+
|      Borough|                Zone|count(1)|
+-------------+--------------------+--------+
|Staten Island|       Arden Heights|       1|
|Staten Island|Eltingville/Annad...|       1|
|    Manhattan|Governor's Island...|       1|
|Staten Island|       Port Richmond|       3|
|Staten Island|         Great Kills|       4|
|        Bronx|       Rikers Island|       4|
|Staten Island|   Rossville/Woodrow|       4|
|     Brooklyn| Green-Wood Cemetery|       4|
|       Queens|         Jamaica Bay|       5|
|Staten Island|         Westerleigh|      12|
|Staten Island|New Dorp/Midland ...|      14|
|Staten Island|       West Brighton|      14|
|Staten Island|             Oakwood|      14|
|        Bronx|        Crotona Park|      14|
|       Queens|       Willets Point|      15|
|       Queens|Breezy Point/Fort...|      16|
|Staten Island|Saint George/New ...|      17|
|       Queens|       Broad Channel|      18|
|Staten Island|     Mariners Harbo

In [13]:
spark.stop()


ConnectionRefusedError: [Errno 111] Connection refused